# Physics-Informed Neural Networks for Unsteady 2D Heat Transfer
## Simulating Meat Cooking on a Weber Grill
---
**Module:** Computational Fluid Dynamics  
**Department:** Mechanical & Aeronautical Engineering  
**Reference:** Raissi, M., Yazdani, A., & Karniadakis, G.E. (2020). *Hidden fluid mechanics.* Science, 367(6481), 1026–1030. https://doi.org/10.1126/science.aaw4741

---
### Learning Objectives
By the end of this notebook you will be able to:
1. Explain how a PINN differs from a standard data-driven ANN
2. Understand how the heat equation is embedded in the PINN loss via automatic differentiation
3. Train both models in PyTorch and interpret training loss curves
4. Benchmark PINN and ANN predictions against FDM and an exact Fourier solution
5. Systematically investigate hyperparameter effects and explain the results
6. Evaluate model generalisation to an unseen material

### Problem Statement
We solve the **2D unsteady heat conduction equation** on a rectangular domain
representing a cross-section of meat on a Weber grill (lid closed):

$$\rho \, c_p \, \frac{\partial T}{\partial t} = k \left(\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}\right)$$

| Quantity | Value |
|---------|-------|
| Domain | $x \in [0,\, 0.05\,\text{m}]$, $y \in [0,\, 0.15\,\text{m}]$, $t \in [0,\, 894\,\text{s}]$ |
| Initial condition | $T(x,y,0) = 288.15\,\text{K}$ (fridge temperature) |
| Boundary conditions | $T = 473.15\,\text{K}$ on all four walls (grill surface) |
| Material inputs | $\rho$, $c_p$, $k$ — vary by meat type |

### Repository Structure
```
pinn-heat-transfer/
├── data/                  ← pre-generated FDM datasets (.npz)
├── utils/
│   ├── config.py          ← all constants (domain, BCs, meat properties)
│   ├── fourier.py         ← exact Fourier series solution
│   ├── dataset.py         ← data loading, normalisation, collocation points
│   ├── models.py          ← HeatNet backbone, ANN, PINN classes
│   ├── training.py        ← train_ann, train_pinn functions
│   └── evaluation.py      ← predict, benchmark, plot helpers
├── generate_dataset.py    ← instructor tool (do not modify)
└── PINN_Heat_Transfer.ipynb  ← this notebook
```


---
## Section 1: Setup

In [ ]:
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

# Import all project utilities from utils/
from utils.config import (
    XMAX, YMAX, T_MAX, NX, NY, NT, DT,
    T_BOUNDARY, T_INITIAL, T_SAFE, DELTA_T,
    MEAT_PROPERTIES, TEST_MEAT,
)
from utils.dataset    import build_ann_dataloader, sample_collocation_points, load_meat_data, normalise_inputs
from utils.models     import ANN, PINN
from utils.training   import train_ann, train_pinn
from utils.evaluation import (evaluate_model, plot_loss_curves,
                               plot_field_comparison, plot_centre_temperature,
                               print_metrics_table)
from utils.fourier    import T_fourier_grid, T_fourier_centre_history

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {device}")
print(f"PyTorch  : {torch.__version__}")

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)


---
## Section 2: Reference Data

The datasets in `data/` were generated from a finite difference method (FDM)
solver using beef, chicken, and pork properties. Each `.npz` file contains
interior spatiotemporal points $(x, y, t, \rho, c_p, k, T)$.

> **Note:** You are not provided with the FDM solver. The datasets are your
> reference ground truth. You will compare your trained models against these
> and against the exact Fourier series solution.


In [ ]:
# ── Load FDM datasets ─────────────────────────────────────────────────────────
DATA_DIR = "data"
fdm_data  = {}

print("Loading FDM reference datasets...")
for meat in MEAT_PROPERTIES:
    raw = load_meat_data(DATA_DIR, meat)
    props = MEAT_PROPERTIES[meat]

    # Reconstruct 3D field (nx, ny, nt) from flat dataset for evaluation
    # Interior only — boundaries are at T_BOUNDARY (already enforced)
    nx_int = NX - 2; ny_int = NY - 2
    n_spatial = nx_int * ny_int

    x_grid = np.linspace(0, XMAX, NX)
    y_grid = np.linspace(0, YMAX, NY)
    t_grid = np.arange(NT) * DT

    # Reshape T back to (nx, ny, nt) including boundaries
    u = np.full((NX, NY, NT), T_BOUNDARY)
    for it in range(NT):
        T_slice = raw['T'][it * n_spatial : (it + 1) * n_spatial]
        u[1:-1, 1:-1, it] = T_slice.reshape(nx_int, ny_int)

    fdm_data[meat] = {
        'u': u, 'x': x_grid, 'y': y_grid, 't': t_grid,
        'rho': props['rho'], 'cp': props['cp'], 'k': props['k'],
    }
    alpha = props['k'] / (props['rho'] * props['cp'])
    print(f"  {meat:8s}: alpha = {alpha:.3e} m²/s | "
          f"T_centre(final) = {u[NX//2, NY//2, -1]:.2f} K")

print("\nReference data loaded for:", list(fdm_data.keys()))


### 2.1 Exact Fourier Series Solution

For this problem configuration (uniform Dirichlet BCs, uniform IC) an exact
analytical solution exists as a double Fourier series:

$$T(x,y,t) = T_s + \sum_{m=1}^{\infty}\sum_{n=1}^{\infty} A_{mn}\,
\sin\!\left(\frac{m\pi x}{L_x}\right)\sin\!\left(\frac{n\pi y}{L_y}\right)
e^{-\alpha\left(\frac{m^2\pi^2}{L_x^2}+\frac{n^2\pi^2}{L_y^2}\right)t}$$

$$A_{mn} = \frac{4(T_i-T_s)}{mn\pi^2}(1-\cos m\pi)(1-\cos n\pi)$$

This is implemented in `utils/fourier.py` and is used as the gold-standard
reference alongside the FDM.


In [ ]:
import matplotlib.pyplot as plt

# ── Visualise FDM reference fields and centre temperature ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (meat, sol) in zip(axes, fdm_data.items()):
    im = ax.contourf(sol['y']*100, sol['x']*100, sol['u'][:,:,-1],
                     30, cmap='hot', vmin=T_INITIAL, vmax=T_BOUNDARY)
    plt.colorbar(im, ax=ax, label='T [K]')
    alpha = sol['k'] / (sol['rho'] * sol['cp'])
    ax.set_title(f"{meat.capitalize()}\n"
                 f"α = {alpha:.2e} m²/s | t = {sol['t'][-1]:.0f} s", fontsize=10)
    ax.set_xlabel('y [cm]'); ax.set_ylabel('x [cm]')
plt.suptitle('FDM Reference — Final Temperature Field', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Centre temperature vs time
plt.figure(figsize=(9, 5))
for meat, sol in fdm_data.items():
    alpha = sol['k'] / (sol['rho'] * sol['cp'])
    T_exact_c = T_fourier_centre_history(sol['t'], alpha)
    plt.plot(sol['t'], sol['u'][NX//2, NY//2, :], linewidth=2,
             label=f"{meat.capitalize()} FDM")
    plt.plot(sol['t'], T_exact_c, '--', linewidth=1.2, alpha=0.7,
             label=f"{meat.capitalize()} Exact")
plt.axhline(T_SAFE, color='red', linestyle=':', linewidth=1.5,
            label=f'Safe temp. {T_SAFE-273.15:.0f}°C', alpha=0.7)
plt.xlabel('Time [s]', fontsize=12); plt.ylabel('Centre Temperature [K]', fontsize=12)
plt.title('Centre Temperature vs Time — FDM vs Exact (Fourier)', fontsize=11)
plt.legend(fontsize=9, ncol=2); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## Section 3: Data Preparation

### 3.1 Input Normalisation
All six network inputs are mapped to $[0, 1]$ before being passed to the network.
This is essential for stable gradient flow — unnormalised inputs with very different
scales cause the loss landscape to be poorly conditioned.

| Input | Physical range | Normalisation |
|-------|---------------|---------------|
| $x$ | $[0,\, L_x]$ | $\hat{x} = x / L_x$ |
| $y$ | $[0,\, L_y]$ | $\hat{y} = y / L_y$ |
| $t$ | $[0,\, t_{\max}]$ | $\hat{t} = t / t_{\max}$ |
| $\rho$ | $[1050,\, 1090]$ kg/m³ | min-max over training meats |
| $c_p$ | $[2720,\, 3300]$ J/kg·K | min-max over training meats |
| $k$ | $[0.412,\, 0.480]$ W/m·K | min-max over training meats |

### 3.2 ANN Training Data
The ANN is trained on FDM data points sampled from the pre-generated datasets.

### 3.3 PINN Collocation Points
The PINN is trained **without any FDM data**. Instead, random points are
sampled across the space-time-material domain to enforce:
- The PDE residual at interior points
- Dirichlet BCs on all four walls
- The initial condition at $t = 0$


In [ ]:
# ── ANN DataLoader ────────────────────────────────────────────────────────────
print("Building ANN training DataLoader...")
loader, n_ann = build_ann_dataloader(
    data_dir    = DATA_DIR,
    meat_names  = list(MEAT_PROPERTIES.keys()),
    n_per_meat  = 5000,
    batch_size  = 512,
    device      = device,
    seed        = 42,
)

# ── PINN Collocation Points ───────────────────────────────────────────────────
print("\nSampling PINN collocation points...")
col = sample_collocation_points(N_col=8000, N_bc=600, N_ic=600, seed=42)


---
## Section 4: Neural Network Architectures

Both the ANN and PINN use **identical network architecture** — a fully-connected
network with `tanh` activations and Xavier (Glorot) weight initialisation.

The **only** difference between them is the loss function.

| Property | ANN | PINN |
|---------|-----|------|
| Architecture | Identical | Identical |
| Loss function | MSE vs FDM data | PDE + BC + IC residuals |
| Training data required | Yes (FDM) | No |
| Physics enforced | No | Yes (via autograd) |

### Why `tanh`?
The PINN computes second-order spatial derivatives of the network output via
automatic differentiation. This requires the activation function to be at least
twice differentiable everywhere. `tanh` satisfies this. `ReLU` does not — its
second derivative is zero almost everywhere.

### Architecture Summary

```
Input layer  :  6 neurons  (x̂, ŷ, t̂, ρ̂, ĉ_p, k̂)
Hidden layers:  N_HIDDEN × N_NEURONS neurons  [tanh]
Output layer :  1 neuron   (T [K])
```

### PINN Loss Function

$$\mathcal{L}_{\text{PINN}} = \lambda_{\text{pde}}\,\mathcal{L}_{\text{pde}} + \lambda_{\text{bc}}\,\mathcal{L}_{\text{bc}} + \lambda_{\text{ic}}\,\mathcal{L}_{\text{ic}}$$

**PDE residual** — enforced at interior collocation points via autograd:

$$\mathcal{L}_{\text{pde}} = \frac{1}{N_c}\sum_{i=1}^{N_c}\left[\frac{\rho c_p \partial T/\partial t - k(\partial^2 T/\partial x^2 + \partial^2 T/\partial y^2)}{\rho c_p \Delta T / t_{\max}}\right]^2$$

**Chain rule for normalised coordinates:**

$$\frac{\partial T}{\partial t} = \frac{\partial T}{\partial \hat{t}} \cdot \frac{1}{t_{\max}}, \qquad \frac{\partial^2 T}{\partial x^2} = \frac{\partial^2 T}{\partial \hat{x}^2} \cdot \frac{1}{L_x^2}$$


---
## Section 5: Training

### 5.1 Hyperparameter Configuration

The block below defines all tunable hyperparameters.  
**In Section 7, you will vary these systematically.** For now, leave them at baseline.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  HYPERPARAMETERS — modify these values for Section 7 experiments
# ═══════════════════════════════════════════════════════════════════════════════

N_HIDDEN    = 5        # Number of hidden layers
N_NEURONS   = 40       # Neurons per hidden layer

LR          = 1e-3     # Adam learning rate
EPOCHS_ANN  = 5000     # Training epochs (ANN)
EPOCHS_PINN = 10000    # Training epochs (PINN — physics is harder to satisfy)

LAMBDA_PDE  = 1.0      # Weight on PDE residual loss
LAMBDA_BC   = 1.0      # Weight on boundary condition loss
LAMBDA_IC   = 1.0      # Weight on initial condition loss

# ═══════════════════════════════════════════════════════════════════════════════

# Instantiate models
ann  = ANN(n_hidden=N_HIDDEN, n_neurons=N_NEURONS).to(device)
pinn = PINN(n_hidden=N_HIDDEN, n_neurons=N_NEURONS,
             lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC,
             lambda_ic=LAMBDA_IC).to(device)

# Print summaries
print("=" * 45)
ann.architecture_summary()
print()
pinn.architecture_summary()
print("=" * 45)


In [ ]:
# ── Train ANN ─────────────────────────────────────────────────────────────────
loss_ann = train_ann(ann, loader, epochs=EPOCHS_ANN, lr=LR, device=device)


In [ ]:
# ── Train PINN ────────────────────────────────────────────────────────────────
loss_pinn = train_pinn(pinn, col, epochs=EPOCHS_PINN, lr=LR, device=device)


In [ ]:
# ── Plot training loss curves ─────────────────────────────────────────────────
plot_loss_curves(loss_ann, loss_pinn,
                 lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC, lambda_ic=LAMBDA_IC)


---
## Section 6: Evaluation and Benchmarking

Each model is evaluated against:
1. **FDM** — finite difference reference (pre-generated dataset)
2. **Exact** — Fourier series analytical solution

Error metrics: Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) in Kelvin.


In [ ]:
# ── Evaluate on all training meats at final time step ─────────────────────────
print("=" * 65)
eval_results = {}
for meat in MEAT_PROPERTIES:
    print()
    r_ann  = evaluate_model(ann,  meat, MEAT_PROPERTIES[meat],
                             fdm_data[meat], device=device)
    r_pinn = evaluate_model(pinn, meat, MEAT_PROPERTIES[meat],
                             fdm_data[meat], device=device)
    eval_results[meat] = {'ann': r_ann, 'pinn': r_pinn}
print("=" * 65)


In [ ]:
# ── Error metric summary table ────────────────────────────────────────────────
print_metrics_table(eval_results)


In [ ]:
# ── Field comparison plots ────────────────────────────────────────────────────
for meat in MEAT_PROPERTIES:
    plot_field_comparison(
        meat,
        eval_results[meat]['ann'],
        eval_results[meat]['pinn'],
        fdm_data[meat]['x'],
        fdm_data[meat]['y'],
    )


In [ ]:
# ── Centre temperature benchmark ─────────────────────────────────────────────
for meat in MEAT_PROPERTIES:
    plot_centre_temperature(
        meat,
        MEAT_PROPERTIES[meat],
        fdm_data[meat],
        models_dict={'ANN': ann, 'PINN': pinn},
        device=device,
    )


---
## Section 7: Hyperparameter Tuning  *(Student Task)*

### Background
Hyperparameters are configuration choices made **before** training that control
the model structure and optimisation process. They are not learned — you must
choose them deliberately.

### Your Task
Run **one hyperparameter at a time** (all others held at baseline values).
For each experiment:
1. Re-run Section 5 with the new value
2. Record the final training loss and MAE vs FDM in `experiment_log`
3. Plot the loss curve
4. In your report, **explain** the result — not just what changed, but *why*

| Hyperparameter | Baseline | Suggested values to test |
|---------------|---------|--------------------------|
| `N_HIDDEN` | 5 | 2, 3, 5, 7, 9 |
| `N_NEURONS` | 40 | 10, 20, 40, 80, 128 |
| `LR` | 1e-3 | 1e-2, 1e-3, 5e-4, 1e-4 |
| `LAMBDA_PDE` | 1.0 | 0.1, 1.0, 5.0, 10.0 |
| `LAMBDA_BC` / `LAMBDA_IC` | 1.0 | 0.1, 1.0, 5.0, 10.0 |

### Report Reflection Questions

Answer each in your report, supported by quantitative results and figures:

**Q1 — Depth vs width:**  
How does increasing `N_HIDDEN` (depth) affect the model differently from
increasing `N_NEURONS` (width)? What does each change about the network's
representational capacity?

**Q2 — Learning rate:**  
What happens when `LR` is too high? Too low? Why does the Adam optimiser
behave more robustly than a fixed learning rate method?

**Q3 — Loss weights (PINN only):**  
Set `LAMBDA_BC = LAMBDA_IC = 0` and retrain. What happens? What does this
reveal about the role of boundary and initial conditions in the PINN framework?

**Q4 — Activation function (PINN only):**  
Replace `tanh` with `ReLU` in the PINN and attempt training. What goes wrong,
and why? *(Hint: examine the second-order derivative in the PDE residual.)*

**Q5 — ANN vs PINN generalisation:**  
Which model performs better on lamb (Section 8 — unseen material)?
Relate your answer to the structure of each model's loss function.


In [ ]:
# ── Experiment runner ─────────────────────────────────────────────────────────
import torch.nn as nn

def run_experiment(model_type='PINN',
                   n_hidden=5, n_neurons=40, lr=1e-3, epochs=5000,
                   lambda_pde=1.0, lambda_bc=1.0, lambda_ic=1.0,
                   activation=None, label=None):
    """
    Train a fresh model with the given hyperparameters and evaluate on beef.

    Parameters
    ----------
    model_type  : 'PINN' or 'ANN'
    n_hidden    : number of hidden layers
    n_neurons   : neurons per hidden layer
    lr          : Adam learning rate
    epochs      : number of training epochs
    lambda_*    : PINN loss weights (ignored for ANN)
    activation  : None (use tanh default) or e.g. nn.ReLU(), nn.Sigmoid()
    label       : descriptive string for logging

    Returns
    -------
    model    : trained model
    history  : loss history dict (PINN) or dict with 'loss' key (ANN)
    metrics  : error metrics dict from evaluate_model()
    """
    if label is None:
        label = f"{model_type} | layers={n_hidden} | neurons={n_neurons} | lr={lr:.0e}"
    print(f"\nExperiment: {label}")
    print("-" * 60)

    if model_type == 'PINN':
        model = PINN(n_hidden=n_hidden, n_neurons=n_neurons,
                      lambda_pde=lambda_pde, lambda_bc=lambda_bc,
                      lambda_ic=lambda_ic).to(device)
    else:
        model = ANN(n_hidden=n_hidden, n_neurons=n_neurons).to(device)

    # Optional: replace default tanh with a custom activation
    if activation is not None:
        for name, child in list(model.named_modules()):
            if isinstance(child, nn.Tanh):
                parts = name.split('.')
                parent = model
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                setattr(parent, parts[-1], activation)
        model._xavier_init()

    if model_type == 'PINN':
        history = train_pinn(model, col, epochs=epochs, lr=lr,
                              print_every=max(1, epochs // 5), device=device)
    else:
        history = train_ann(model, loader, epochs=epochs, lr=lr,
                             print_every=max(1, epochs // 5), device=device)

    result = evaluate_model(model, 'beef', MEAT_PROPERTIES['beef'],
                             fdm_data['beef'], device=device)
    return model, history, result['metrics']


# ── Experiment log ────────────────────────────────────────────────────────────
# Append a dict for each experiment: {'label', 'mae_fdm', 'rmse_fdm', 'notes'}
experiment_log = []

def log_experiment(label, metrics, notes=''):
    """Helper to record an experiment result."""
    experiment_log.append({
        'label':    label,
        'mae_fdm':  metrics['mae_fdm'],
        'rmse_fdm': metrics['rmse_fdm'],
        'notes':    notes,
    })
    print(f"Logged: {label}")


def print_experiment_log():
    """Print all recorded experiments as a formatted table."""
    if not experiment_log:
        print("No experiments recorded yet.")
        return
    header = f"{'Label':<45} {'MAE (FDM)':>10} {'RMSE (FDM)':>11}  Notes"
    print(header)
    print("-" * len(header))
    for r in experiment_log:
        print(f"{r['label']:<45} {r['mae_fdm']:>10.4f} {r['rmse_fdm']:>10.4f}  "
              f"{r.get('notes', '')}")


# ── Example experiment (uncomment to run) ────────────────────────────────────
# model_exp1, hist_exp1, met_exp1 = run_experiment(
#     model_type='PINN', n_hidden=3, n_neurons=20, lr=1e-3, epochs=5000,
#     label='Shallow PINN: 3 layers × 20 neurons')
# log_experiment('Shallow PINN 3×20', met_exp1, notes='Under-parameterised')

print("Experiment runner ready.")
print("Use run_experiment() to test hyperparameter configurations.")
print("Use log_experiment() to record results, print_experiment_log() to view.")


In [ ]:
# ── Print experiment results table ────────────────────────────────────────────
print_experiment_log()


---
## Section 8: Test Meat — Generalisation Evaluation  *(Student Task)*

Your models were trained on **beef, chicken, and pork**. Now evaluate your
best models on **lamb** — a material never seen during training.

This tests **generalisation**: can the PINN extrapolate across the material
property space, or does it only memorise the training distribution?

### Your Task
1. Substitute your best-performing models from Section 7 for `best_ann` and `best_pinn`
2. Evaluate both on lamb
3. Compare the generalisation error (MAE vs Exact) to the training meat errors
4. In your report, explain which model generalises better and why — relate your
   answer to the structure of each model's loss function


In [ ]:
# ── Load test meat FDM reference ──────────────────────────────────────────────
print("Loading test meat reference data...")
raw_test = load_meat_data(DATA_DIR, 'lamb')
props_lamb = TEST_MEAT['lamb']
nx_int = NX - 2; ny_int = NY - 2; n_spatial = nx_int * ny_int

x_grid = np.linspace(0, XMAX, NX)
y_grid = np.linspace(0, YMAX, NY)
t_grid = np.arange(NT) * DT

u_lamb = np.full((NX, NY, NT), T_BOUNDARY)
for it in range(NT):
    u_lamb[1:-1, 1:-1, it] = raw_test['T'][it*n_spatial:(it+1)*n_spatial].reshape(nx_int, ny_int)

fdm_lamb = {'u': u_lamb, 'x': x_grid, 'y': y_grid, 't': t_grid, **props_lamb}

alpha_lamb = props_lamb['k'] / (props_lamb['rho'] * props_lamb['cp'])
print(f"Lamb: alpha = {alpha_lamb:.3e} m²/s | "
      f"T_centre(final) = {u_lamb[NX//2, NY//2, -1]:.2f} K")


In [ ]:
# ── Evaluate on test meat ─────────────────────────────────────────────────────
# Replace 'ann' and 'pinn' below with your best models from Section 7
best_ann  = ann    # TODO: replace with your optimised model
best_pinn = pinn   # TODO: replace with your optimised model

print("ANN on test meat (lamb):")
r_ann_lamb = evaluate_model(best_ann, 'lamb', props_lamb, fdm_lamb, device=device)

print()
print("PINN on test meat (lamb):")
r_pinn_lamb = evaluate_model(best_pinn, 'lamb', props_lamb, fdm_lamb, device=device)


In [ ]:
# ── Field comparison ──────────────────────────────────────────────────────────
plot_field_comparison('lamb', r_ann_lamb, r_pinn_lamb, x_grid, y_grid)


In [ ]:
# ── Centre temperature benchmark ─────────────────────────────────────────────
plot_centre_temperature(
    'lamb', props_lamb, fdm_lamb,
    models_dict={'ANN': best_ann, 'PINN': best_pinn},
    device=device,
)


---
## Section 9: Report Guidance

Your report should be structured around the following, supported by figures
and quantitative results from this notebook.

### 1. Model Understanding
- Describe the network architecture. What is the role of each hyperparameter?
- Derive and explain the chain rule used to compute physical derivatives from
  normalised-coordinate autograd outputs
- Why is `tanh` required for PINN autograd? What fails with `ReLU`?
- Explain the composite PINN loss — what does each term enforce?

### 2. Hyperparameter Study
- Present all experiments in a table (use `print_experiment_log()`)
- For each hyperparameter: describe the trend and provide a mathematical or
  physical explanation
- Identify the configuration that minimises MAE vs FDM and MAE vs Exact

### 3. Benchmarking — Training Meats
- Compare ANN, PINN, FDM, and Exact quantitatively (MAE, RMSE) for all three
  training meats
- Plot temperature fields and centre temperature histories
- Discuss: in what regimes (early time, late time, boundary regions) does each
  model perform best or worst?

### 4. Generalisation — Test Meat (Lamb)
- Compare ANN and PINN on the unseen test material
- Which generalises better? Why?

### 5. Critical Method Comparison

| Criterion | FDM | ANN | PINN |
|-----------|-----|-----|------|
| Requires simulation data | ✓ | ✓ | ✗ |
| Enforces physics | ✓ | ✗ | ✓ |
| Generalises across materials | Re-run per material | Partially | ✓ |
| Computational cost (inference) | High | Low | Low |
| Interpretability | High | Low | Medium |

Discuss this table in the context of your experimental results. Are there cases
where the ANN outperforms the PINN? What does this reveal about the limitations
of each approach?

---
*Department of Mechanical & Aeronautical Engineering | University of Pretoria*
